# Entrenamiento con el train completo (~1.93M imágenes) — Colab

Notebook delgado: todo el entrenamiento vive en `src/entrenamiento` (el mismo código de la prueba local de 200k).
Acá solo se prepara la máquina y se lanza `scripts/entrenar.py --config configs/completo_2m.json`.

**Para empezar o retomar: Ejecutar todo.** La corrida guarda todo en Drive (`carpeta_corrida`) y sigue sola desde
el último checkpoint. Si la sesión se corta, se reconecta y se vuelve a ejecutar todo: se vuelven a bajar los
parquets (el disco de la máquina se borra) y el entrenamiento retoma donde quedó.

**Dos corridas en la misma sesión:** cuando termina una, cambiar `CONFIG` en la celda de parámetros, ejecutarla
y ejecutar solo la celda de entrenamiento (sección 4): los datos ya están en el disco y no se vuelven a bajar.

**Para pausar a mano:** interrumpir la celda de entrenamiento (■). El script recibe la señal, guarda un checkpoint
y sale. `MAX_HORAS` hace lo mismo solo, antes de que Colab corte la sesión.

**Datos:** al empezar cada sesión se bajan los parquets de train y val y se **recomprimen a 384 px (JPEG calidad
90)** en el disco de la máquina: de ~181 GB a ~77 GB, para que entren en los ~121 GB de la L4. El modelo sigue
viendo 224 px con la misma augmentation: el RandomResizedCrop (60-100% del área) y el Resize de validación siempre
achican, y el recorte más chico de una imagen de 384 px (~297 px) también. Se conservan el orden de las filas y las
demás columnas, así que los splits y el sampler no cambian.

Por qué no se leen de HF de a uno durante el entrenamiento: cada ronda es balanceada por celda, así que toca casi
todos los parquets; leerlos de HF en cada ronda sería bajar ~180 GB por ronda.

In [ ]:
# ---- Parámetros de la sesión ----
import os
RAMA = "main"
# Qué se entrena. La carpeta de la corrida en Drive lleva el nombre de la config.
#   configs/completo_2m.json        celdas K=70 + focal haversine + SAM + augmentation + snapshots (ya entrenado)
#   configs/baseline_celdas_2m.json celdas K=70, CE simple, sin augmentation, sin SAM, sin snapshots
#   configs/baseline_paises_2m.json 71 países (>= 1000 filas, como el de 20k), CE simple, sin augmentation/SAM/snapshots
#   configs/baseline_celdas_aug_2m.json   celdas K=70, CE simple + augmentation (RandomResizedCrop + ColorJitter), sin SAM ni snapshots
#   configs/baseline_paises101_2m.json    101 países (>= 50 imágenes en el train de HF, como COLABstreetview_country_resnet18_2), CE simple
CONFIG = "configs/baseline_paises101_2m.json"
DRIVE = "/content/drive/MyDrive/geo-project"     # Carpeta de Drive del proyecto.
INSUMOS_DRIVE = f"{DRIVE}/insumos"                # manifest.parquet, splits.parquet, validation.parquet (subidos a mano).
CARPETA_CORRIDA = f"{DRIVE}/entrenamiento_2m/{os.path.splitext(os.path.basename(CONFIG))[0]}"
MAX_HORAS = 20.0    # Por sesión. Por debajo del límite de Colab (24 h en Pro) para que guarde y salga solo.
REPO = "/content/geo-project"
SHARDS = "/content/shards"                        # Disco local de la máquina (rápido, se borra al cerrar la sesión).

## 1. Máquina

In [ ]:
import os, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPUs:", os.cpu_count())
!free -g | head -2
total, usado, libre = shutil.disk_usage("/content")
# Si la sesión ya recomprimió parte de los datos (se relanzó sin reiniciar), eso cuenta como espacio disponible.
hechos = sum(e.stat().st_size for e in os.scandir(f"{SHARDS}/data") if e.is_file()) if os.path.isdir(f"{SHARDS}/data") else 0
print(f"Disco /content: {libre / 1e9:.0f} GB libres de {total / 1e9:.0f} GB ({hechos / 1e9:.0f} GB ya recomprimidos)")
assert libre + hechos > 100e9, "Hacen falta ~80 GB para los parquets recomprimidos de train y val (+ margen)." 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CARPETA_CORRIDA, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    assert os.path.exists(f"{INSUMOS_DRIVE}/{nombre}"), f"Falta subir {nombre} a {INSUMOS_DRIVE}" 

## 2. Código

In [ ]:
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} pull -q --ff-only
else:
    !git clone -q --branch {RAMA} https://github.com/gggelme/geo-project.git {REPO}
!git -C {REPO} log -1 --oneline

## 3. Datos al disco local

In [ ]:
# Insumos chicos: de Drive al disco local (leerlos de Drive en cada ronda sería lento).
INSUMOS = "/content/insumos"
os.makedirs(INSUMOS, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    if not os.path.exists(f"{INSUMOS}/{nombre}"):
        shutil.copy(f"{INSUMOS_DRIVE}/{nombre}", f"{INSUMOS}/{nombre}")
print(os.listdir(INSUMOS))

In [ ]:
# Baja cada parquet de train y val (no los de test) en la revisión fijada en la config, lo recomprime a 384 px y
# borra el original. ~30-60 min. Si se corta, volver a ejecutar: saltea los que ya están hechos.
import json
try:  # Token de HF desde los Secrets de Colab (el script corre en otro proceso y lo toma de la variable).
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("Sin HF_TOKEN en Secrets: se baja sin token (puede ser más lento).")
revision = json.load(open(f"{REPO}/{CONFIG}"))["hf_revision"]
procesos = max(1, (os.cpu_count() or 2) - 1)
!cd {REPO} && python scripts/recomprimir_shards.py --destino {SHARDS}/data --tmp /content/shards_tmp --splits {INSUMOS}/splits.parquet --revision {revision} --procesos {procesos}
total, usado, libre = shutil.disk_usage("/content")
print(f"Disco /content: {libre / 1e9:.0f} GB libres")

## 4. Entrenamiento

Retoma solo si `CARPETA_CORRIDA` ya tiene checkpoints. La salida se ve en vivo; además todo queda en
`log.jsonl` e `historial.csv` dentro de la carpeta de la corrida.

In [ ]:
import signal, sys
num_workers = max(2, (os.cpu_count() or 2) - 1)
comando = [sys.executable, "scripts/entrenar.py", "--config", CONFIG, "--cambiar",
           f"carpeta_corrida={CARPETA_CORRIDA}", f"datos_local={SHARDS}/data",
           f"manifest={INSUMOS}/manifest.parquet", f"splits={INSUMOS}/splits.parquet",
           f"val_global={INSUMOS}/validation.parquet", f"num_workers={num_workers}", f"max_horas={MAX_HORAS}"]
print(" ".join(comando))
proceso = subprocess.Popen(comando, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for linea in proceso.stdout:
        print(linea, end="")
except KeyboardInterrupt:
    # SIGTERM: el script termina el paso actual, guarda un checkpoint y sale (puede tardar si está evaluando).
    print("Interrumpido: se pide al script que guarde y salga (esperar, no interrumpir de nuevo)...")
    proceso.send_signal(signal.SIGTERM)
    while proceso.poll() is None:
        try:
            for linea in proceso.stdout:
                print(linea, end="")
        except KeyboardInterrupt:
            print("Sigue guardando; esperando a que salga...")
codigo = proceso.wait()
print({0: "TERMINADO", 3: "PAUSADO: volver a ejecutar todo para seguir"}.get(codigo, f"ERROR (código {codigo})"))

## 5. Cómo va

In [ ]:
import pandas as pd
ruta = f"{CARPETA_CORRIDA}/historial.csv"
if os.path.exists(ruta):
    h = pd.read_csv(ruta)
    display(h[["ronda", "loss_train", "val_ce_haversine", "val_top1", "val_top5", "val_km_mediana",
               "valglobal_top1", "valglobal_km_mediana", "lr_cabeza", "horas_acumuladas"]].tail(15))